In [5]:
import calc_series_change
import pandas as pd
import pyodbc
from datetime import datetime, timedelta
import time
import numpy as np
import sys
import alpha_table
import psycopg2
import moex_cls as mcls

user = 'tmp_skosyrkov'
password = 'A65h9bmstu12sg9!'

def compose_otchet(inp_period):
    start_date = datetime.today() - timedelta(days=365 * 2 - inp_period)
    finish_date = datetime.today() - timedelta(days=inp_period)
    con = pyodbc.connect(
        "driver={SQL Server};"
        "server=AMDBRATES\DBMVTBAM01;"
        "database=Rates;"
        "trusted_connection=yes"
    )
    con_dl = psycopg2.connect(
        host="SMSK01DB749",
        port="15432",
        database="WIM_datalake_prod",
        user=user,
        password=password,
    )

    date_str = start_date.strftime('%d.%m.%Y')

    funds = pd.read_sql(
        f"""
        SELECT DISTINCT period, naimenovanie, stoimost_paya
        FROM pif.stoimost_chistykh_aktivov costs
        INNER JOIN pif.pif nams ON costs.pif = nams.ssylka
        WHERE costs.period >= TO_DATE('{date_str}', 'DD.MM.YYYY')
        ORDER BY period DESC
        """,
        con=con_dl
    )

    isin_and_bench = {
        'БПИФРФИ "ВИМ - Индекс МосБиржи"': "MCF2TR",
        'ОПИФРФИ "ВИМ - Акции"': "MCFTR",
        'ОПИФРФИ "ВИМ - Перспективные размещения"': "VTBFG0013",
        'ОПИФРФИ "ВИМ - Денежный рынок. Рубли"': "RUSFAR",
        'ОПИФРФИ "ВИМ - Казначейский"': "RUCBTRNS",
        'ОПИФРФИ "ВИМ - Металлургия"': "MOEXMM",
        'ОПИФРФИ "ВИМ - Нефтегазовый сектор"': "MOEXOG",
        'ОПИФРФИ "ВИМ - Облигации. Ответственные инвестиции"': "RUCBTRNS",
        'ОПИФРФИ "ВИМ - Сбалансированные инвестиции"': "VTBFG0005",
        'ОПИФРФИ "ВИМ - Сбалансированный"': "VTBFG0006",
        'ОПИФРФИ "ВИМ - Сбалансированный. Консервативный"': "VTBFG0011",
        'ОПИФРФИ "ВИМ - Инновационный"': "MOEXIT",
        'ОПИФРФИ "ВИМ - Накопительный резерв"': "VTBFG0010",
        'ОПИФРФИ "ВИМ - Облигации. Российские эмитенты"': "RUCBTRNS",
        'ОПИФРФИ "ВИМ - Акции. Российские эмитенты"': "MCFTR",
        'БПИФРФИ "ВИМ - Индекс МосБиржи"': "VTBFG0012",
        'БПИФРФИ "Ликвидность"': "RUSFAR",
        'БПИФРФИ "ВИМ - Корпоративные облигации"': "RUCBTRNS",
        'БПИФРФИ "ВИМ - Устойчивое развитие российских компаний"': "MRSVRT",
        # добавляются новые. Чтобы добавить каждый новый, нужно указать как 'нейм' бенча, так и точное название фонда в даталейке 
        'ИПИФФИ "2х ОФЗ"': "VTBST1",
        'ИПИФФИ "Валютное плечо"': "VTBST2",
        'ОПИФРФИ "ВИМ - Умный капитал"': "LQDT",
        'ОПИФРФИ "ВИМ - Накопительный в юанях"': "RUCNYTR",
        'ОПИФРФИ "ВИМ - Ключевой+"': "RUFLCBTR",
        'ОПИФРФИ "ВИМ - Новые валютные облигации"': "RUEU10"
    }

    all_funds_perform = pd.DataFrame()

    # --- ДОБАВЛЯЕМ ОЧИСТКУ ДАННЫХ ПЕРЕД ЦИКЛОМ ---
    funds = funds.dropna(subset=["period", "stoimost_paya"])
    funds = funds.drop_duplicates(subset=["naimenovanie", "period"], keep="last")
    # funds["isin"] = funds["isin"].str.extract(r"(RU[0-9A-Z]{10})", expand=False)
    # ---------------------------------------------

    def fetch_micex_index(security_id, start, finish):
        df = pd.read_sql(
            f""" 
            select TRADEDATE, [CLOSE] as [{security_id}] from rates.dbo.dbo_MICEX_Index
            where securityid = ? and tradedate >= CONVERT(date, ?, 104) and tradedate <= CONVERT(date, ?, 104)
            order by TRADEDATE
            """,
            con=con,
            params=[security_id, start, finish],
        )
        return df.set_index("TRADEDATE").squeeze()

    for one, one_bench in isin_and_bench.items():
        # 1. Фильтруем данные
        sub_df = funds[funds["naimenovanie"] == one]

        # 2. ПРОВЕРКА: если данных для этого ISIN нет, пропускаем
        if sub_df.empty:
            print(
                f"Предупреждение: для {one} не найдено данных в датафрейме funds. Пропускаем."
            )
            continue

        # Сортируем строго по дате перед расчетом
        sub_df = sub_df.sort_values("period")

        # 3. Если данные есть, спокойно делаем расчет
        one_series = calc_series_change.value_growth(
            sub_df.set_index("period")["stoimost_paya"]
        )
        all_funds_perform[one] = one_series

        # Границы дат для текущего фонда
        start_dt = all_funds_perform[one].first_valid_index()
        finish_dt = all_funds_perform[one].last_valid_index()

        # теперь грузим бенч (сразу рядом). match используется т.к. есть кастомные индексы и я их считаю на лету
        try:
            match one_bench:
                case "VTBFG0013":
                    b_df = fetch_micex_index("MIPOTR", start_dt, finish_dt)
                    all_funds_perform[one_bench] = mcls.series_index(
                        pairs=[(b_df, 1)],
                        start_date=start_dt,
                        finish_date=finish_dt,
                        name=one_bench,
                    )

                case "VTBFG0005":
                    b_df_1 = fetch_micex_index("MCFTR", start_dt, finish_dt)
                    b_df_2 = fetch_micex_index("RUCBTRNS", start_dt, finish_dt)
                    this_bench_data = (
                        mcls.quarterly_index(
                            pairs=[(b_df_1, 0.5), (b_df_2, 0.5)],
                            start_date=start_dt,
                            finish_date=finish_dt,
                            name=one_bench,
                            cumulative_yield=True
                        )
                        .reset_index()[["date", one_bench]]
                        .set_index("date")[one_bench]
                    )  # функция возвращает большой датафрейм (
                    # полезно для проверки). Я переделываю датафрейм в series чтобы можно было легко вязать все остальное
                    all_funds_perform[one_bench] = this_bench_data

                case "VTBFG0006":
                    b_df_1 = fetch_micex_index("MCFTR", start_dt, finish_dt)
                    b_df_2 = fetch_micex_index("RUCBTRNS", start_dt, finish_dt)
                    all_funds_perform[one_bench] = (
                        mcls.quarterly_index(
                            pairs=[(b_df_1, 0.65), (b_df_2, 0.35)],
                            start_date=start_dt,
                            finish_date=finish_dt,
                            name=one_bench,
                            cumulative_yield=True
                        )
                        .reset_index()[["date", one_bench]]
                        .set_index("date")[one_bench]
                    )  # функция возвращает большой датафрейм (
                    # полезно для проверки). Я переделываю датафрейм в series чтобы можно было легко вязать все остальное
                    all_funds_perform[one_bench] = this_bench_data

                case "VTBFG0011":
                    b_df_1 = fetch_micex_index("MCFTR", start_dt, finish_dt)
                    b_df_2 = fetch_micex_index("RUCBTRNS", start_dt, finish_dt)
                    all_funds_perform[one_bench] = (
                        mcls.quarterly_index(
                            pairs=[(b_df_1, 0.2), (b_df_2, 0.8)],
                            start_date=start_dt,
                            finish_date=finish_dt,
                            name=one_bench,
                            cumulative_yield=True
                        )
                        .reset_index()[["date", one_bench]]
                        .set_index("date")[one_bench]
                    )  # функция возвращает большой датафрейм (
                    # полезно для проверки). Я переделываю датафрейм в series чтобы можно было легко вязать все остальное
                    all_funds_perform[one_bench] = this_bench_data

                case "VTBFG0010":
                    b_df_1 = fetch_micex_index("MCFTR", start_dt, finish_dt)
                    b_df_2 = fetch_micex_index("RUCBTRNS", start_dt, finish_dt)
                    all_funds_perform[one_bench] = (
                        mcls.quarterly_index(
                            pairs=[(b_df_1, 0.4), (b_df_2, 0.6)],
                            start_date=start_dt,
                            finish_date=finish_dt,
                            name=one_bench,
                            cumulative_yield=True
                        )
                        .reset_index()[["date", one_bench]]
                        .set_index("date")[one_bench]
                    )  # функция возвращает большой датафрейм (
                    # полезно для проверки). Я переделываю датафрейм в series чтобы можно было легко вязать все остальное
                    all_funds_perform[one_bench] = this_bench_data

                case "VTBFG0017":
                    b_df_1 = fetch_micex_index("IMOEX", start_dt, finish_dt)
                    b_df_2 = fetch_micex_index("RUCBTRNS", start_dt, finish_dt)
                    all_funds_perform[one_bench] = (
                        mcls.quarterly_index(
                            pairs=[(b_df_1, 0.5), (b_df_2, 0.5)],
                            start_date=start_dt,
                            finish_date=finish_dt,
                            name=one_bench,
                            cumulative_yield=True
                        )
                        .reset_index()[["date", one_bench]]
                        .set_index("date")[one_bench]
                    )  # функция возвращает большой датафрейм (
                    # полезно для проверки). Я переделываю датафрейм в series чтобы можно было легко вязать все остальное
                    all_funds_perform[one_bench] = this_bench_data

                case "VTBFG0012":
                    b_df = fetch_micex_index("MCF2TR", start_dt, finish_dt)
                    all_funds_perform[one_bench] = mcls.series_index(
                        pairs=[(b_df, 1)],
                        start_date=start_dt,
                        finish_date=finish_dt,
                        name=one_bench,
                    )

                case "VTBST1":
                    b_df = fetch_micex_index("MCF2TR", start_dt, finish_dt)
                    all_funds_perform[one_bench] = mcls.series_index(
                        pairs=[(b_df, 1)],
                        start_date=start_dt,
                        finish_date=finish_dt,
                        name=one_bench,
                    )

                case "VTBFG00228":
                    b_df_1 = fetch_micex_index("RUCNYTR", start_dt, finish_dt)
                    b_df_2 = fetch_micex_index("RUSFARCNY", start_dt, finish_dt)
                    all_funds_perform[one_bench] = (
                        mcls.quarterly_index(
                            pairs=[(b_df_1, 0.5), (b_df_2, 0.5)],
                            start_date=start_dt,
                            finish_date=finish_dt,
                            name=one_bench,
                            cumulative_yield=True
                        )
                        .reset_index()[["date", one_bench]]
                        .set_index("date")[one_bench]
                    )  # функция возвращает большой датафрейм (
                    # полезно для проверки). Я переделываю датафрейм в series чтобы можно было легко вязать все остальное
                    all_funds_perform[one_bench] = this_bench_data

                case _:
                    # Для стандартных бенчмарков, которые загружаются напрямую целиком
                    one_bench_df = fetch_micex_index(one_bench, start_dt, finish_dt)
                    if one_bench == "RUSFAR":
                        all_funds_perform[one_bench] = mcls.series_index(
                            pairs=[(one_bench_df, 1)],
                            start_date=start_dt,
                            finish_date=finish_dt,
                            name=one_bench,
                            cumulative=True,
                        )
                    elif one_bench == "RUSFARCNY":
                        all_funds_perform[one_bench] = mcls.series_index(
                            pairs=[(one_bench_df, 1)],
                            start_date=start_dt,
                            finish_date=finish_dt,
                            name=one_bench,
                            cumulative=True,
                        )
                    else:
                        all_funds_perform[one_bench] = mcls.series_index(
                            pairs=[(one_bench_df, 1)],
                            start_date=start_dt,
                            finish_date=finish_dt,
                            name=one_bench,
                        )

        except Exception as e:
            print(f"Ошибка при обработке бенчмарка {one_bench} для фонда {one}: {e}")
    return all_funds_perform
x = compose_otchet(1)

<>:20: SyntaxWarning: invalid escape sequence '\D'
<>:20: SyntaxWarning: invalid escape sequence '\D'
C:\Users\tmp_SKosyrkov\AppData\Local\Temp\ipykernel_6172\1186216366.py:20: SyntaxWarning: invalid escape sequence '\D'
  "server=AMDBRATES\DBMVTBAM01;"
C:\Users\tmp_SKosyrkov\AppData\Local\Temp\ipykernel_6172\1186216366.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  funds = pd.read_sql(
C:\Users\tmp_SKosyrkov\AppData\Local\Temp\ipykernel_6172\1186216366.py:83: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
C:\Users\tmp_SKosyrkov\AppData\Local\Temp\ipykernel_6172\1186216366.py:83: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection)

Ошибка при обработке бенчмарка VTBFG0005 для фонда ОПИФРФИ "ВИМ - Сбалансированные инвестиции": quarterly_index() got an unexpected keyword argument 'cumulative_yield'
Ошибка при обработке бенчмарка VTBFG0006 для фонда ОПИФРФИ "ВИМ - Сбалансированный": quarterly_index() got an unexpected keyword argument 'cumulative_yield'
Ошибка при обработке бенчмарка VTBFG0011 для фонда ОПИФРФИ "ВИМ - Сбалансированный. Консервативный": quarterly_index() got an unexpected keyword argument 'cumulative_yield'
Ошибка при обработке бенчмарка VTBFG0010 для фонда ОПИФРФИ "ВИМ - Накопительный резерв": quarterly_index() got an unexpected keyword argument 'cumulative_yield'


C:\Users\tmp_SKosyrkov\AppData\Local\Temp\ipykernel_6172\1186216366.py:83: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
C:\Users\tmp_SKosyrkov\AppData\Local\Temp\ipykernel_6172\1186216366.py:83: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
C:\Users\tmp_SKosyrkov\AppData\Local\Temp\ipykernel_6172\1186216366.py:83: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
C:\Users\tmp_SKosyrkov\AppData\Local\Temp\ipykernel_6172\1186216366.py:83: UserWarning: pandas only supports SQL

Ошибка при обработке бенчмарка VTBST2 для фонда ИПИФФИ "Валютное плечо": VTBST2: ряд пуст после удаления NaN
Ошибка при обработке бенчмарка LQDT для фонда ОПИФРФИ "ВИМ - Умный капитал": LQDT: ряд пуст после удаления NaN


C:\Users\tmp_SKosyrkov\AppData\Local\Temp\ipykernel_6172\1186216366.py:83: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
C:\Users\tmp_SKosyrkov\AppData\Local\Temp\ipykernel_6172\1186216366.py:83: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
C:\Users\tmp_SKosyrkov\AppData\Local\Temp\ipykernel_6172\1186216366.py:83: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
C:\Users\tmp_SKosyrkov\AppData\Local\Temp\ipykernel_6172\1186216366.py:83: UserWarning: pandas only supports SQL

In [6]:
x.to_excel("test_x.xlsx")